# 🚀 ML Toolkit in Kaggle - Complete Guide

This notebook shows you how to use the enterprise-grade **ML Toolkit** library directly in Kaggle to evaluate and compare multiple machine learning models.

## What You'll Learn:
- ✅ Install the ML Toolkit package in Kaggle
- 📊 Load and prepare data from Kaggle datasets
- 🤖 Configure multiple ML models with preprocessing
- 📈 Compare models with automated evaluation
- 🎨 Visualize performance and feature importance
- 💾 Export results and best models

Let's get started! 🎯

## Section 1️⃣: Install and Import Dependencies

In [ ]:
# Step 1: Install ML Toolkit from GitHub
import subprocess
import sys

print("⏳ Installing ML Toolkit...")
subprocess.check_call([sys.executable, "-m", "pip", "install", 
                       "git+https://github.com/LAFFI01/MY_ML_TOOLS.git", 
                       "-q"])
print("✅ ML Toolkit installed successfully!\n")

In [ ]:
# Step 2: Import all required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE

# Import our ML Toolkit
from my_ml_toolkit import evaluate_and_plot_models

print("✅ All libraries imported successfully!")

# Configure Kaggle paths
INPUT_PATH = "/kaggle/input"
OUTPUT_PATH = "/kaggle/output"

print(f"\n📁 Kaggle Input Path: {INPUT_PATH}")
print(f"📁 Kaggle Output Path: {OUTPUT_PATH}")

# List available datasets
if os.path.exists(INPUT_PATH):
    datasets = os.listdir(INPUT_PATH)
    print(f"\n📦 Available datasets ({len(datasets)}):")
    for dataset in datasets[:10]:  # Show first 10
        print(f"   • {dataset}")

## Section 2️⃣: Load Data from Kaggle Datasets

**Change the dataset paths below to match your Kaggle competition/dataset!**

In [ ]:
# Example: Load Titanic dataset (common Kaggle dataset)
# For other datasets, change the file path accordingly

# Option 1: If you have a built-in dataset
try:
    df = pd.read_csv(f"{INPUT_PATH}/titanic/train.csv")
    print("✅ Titanic dataset loaded!")
except:
    # Option 2: Create sample data for demonstration
    print("⚠️  Using sample data for demonstration")
    df = pd.read_csv(f"{INPUT_PATH}/../../../../../tmp/titanic.csv") if os.path.exists(f"{INPUT_PATH}/../../../../../tmp/titanic.csv") else None
    
    if df is None:
        # Create synthetic data for demo
        np.random.seed(42)
        df = pd.DataFrame({
            'PassengerId': range(1, 892),
            'Age': np.random.normal(30, 15, 891),
            'Fare': np.random.exponential(30, 891),
            'Pclass': np.random.choice([1, 2, 3], 891),
            'Sex': np.random.choice(['male', 'female'], 891),
            'Survived': np.random.choice([0, 1], 891)
        })
        print("📊 Using synthetic Titanic-like data")

# Display dataset info
print(f"\n📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n🔍 First few rows:")
print(df.head())
print(f"\n📋 Data Types:")
print(df.dtypes)
print(f"\n⚠️ Missing Values:")
print(df.isnull().sum())
print(f"\n📈 Dataset Info:")
print(df.describe())

## Section 3️⃣: Prepare Features and Target Variable

In [ ]:
# Step 1: Handle missing values
print("🔧 Data Preprocessing:")
print("=" * 60)

# Fill numeric missing values with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)
        print(f"✓ Filled {col} with median")

# Fill categorical missing values with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)
        print(f"✓ Filled {col} with mode")

print("\n✅ No more missing values!")
print(f"Missing values remaining: {df.isnull().sum().sum()}")

# Step 2: Select features and target
# IMPORTANT: Modify these based on your dataset!
# For Titanic example:
FEATURE_COLS = ['Age', 'Fare', 'Pclass']  # Change this!
TARGET_COL = 'Survived'                    # Change this!

# Filter to only numeric features for simplicity
X_data = df[FEATURE_COLS].select_dtypes(include=[np.number]).fillna(0)
y_data = df[TARGET_COL]

print(f"\n📊 Features (X): {X_data.shape}")
print(f"🎯 Target (y): {y_data.shape}")
print(f"Feature names: {list(X_data.columns)}")
print(f"Target distribution:\n{y_data.value_counts()}")

# Convert to DataFrame with feature names
X = pd.DataFrame(X_data, columns=X_data.columns)
y = pd.Series(y_data)

print(f"\n✅ Data preparation complete!")

## Section 4️⃣: Configure Models and Preprocessing Pipeline

In [ ]:
# Step 1: Define ML Models to Compare
print("🤖 Configuring ML Models:")
print("=" * 60)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf', random_state=42)
}

for model_name in models:
    print(f"✓ {model_name}")

# Step 2: Create Preprocessing Pipeline
print("\n🔧 Preprocessing Pipeline:")
print("=" * 60)

preprocessor = StandardScaler()
print("✓ StandardScaler for feature scaling")

# Step 3: (Optional) Hyperparameter Tuning Grids
param_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1, 10],
        "model__penalty": ['l2']
    },
    "Random Forest": {
        "model__n_estimators": [50, 100],
        "model__max_depth": [5, 10, None]
    },
    "Gradient Boosting": {
        "model__learning_rate": [0.01, 0.1],
        "model__n_estimators": [50, 100]
    }
}

print("\n✓ Hyperparameter grids configured for tuning")

# Step 4: (Optional) Handle Imbalanced Data
# Only use if your target classes are highly imbalanced
sampler = SMOTE(random_state=42)  # or set to None if data is balanced
print("✓ SMOTE sampler configured for imbalanced data handling")

print("\n✅ All models and pipelines configured!")

## Section 5️⃣: Run Model Evaluation and Compare Results

**This is where the ML Toolkit does all the heavy lifting!** ✨

In [ ]:
# RUN THE EVALUATION 🚀
print("\n" + "="*70)
print("🚀 STARTING MODEL EVALUATION WITH ML TOOLKIT")
print("="*70 + "\n")

# Call the main evaluation function
results = evaluate_and_plot_models(
    models=models,
    preprocess_pipeline=preprocessor,
    X=X,  # Features
    y=y,  # Target
    test_size=0.2,                    # 20% test set
    task_type="classification",       # For regression, use "regression"
    target_names=["No", "Yes"],       # Optional: class names
    feature_names=list(X.columns),    # Feature names for importance
    param_grids=param_grids,          # Hyperparameter grids for tuning
    sampler=sampler,                  # Handle imbalanced data
    cv=5,                             # 5-fold cross-validation
    search_type="grid",               # Grid search for tuning
    primary_metric="accuracy",        # Metric to optimize
    plot_lc=True,                     # Plot learning curves
    plot_diagnostics=True,            # Plot confusion matrices
    plot_importance=True,             # Show feature importance
    plot_comparison=True,             # Compare models
    verbose=True                      # Show detailed output
)

print("\n✅ Evaluation complete! Results saved to 'results' dictionary.")

## Section 6️⃣: Analyze Feature Importance and Model Performance

In [ ]:
# Extract key results from the evaluation
summary_df = results["summary_df"]
best_models = results["best_models"]
raw_cv_scores = results["raw_cv_scores"]
ultimate_winner = results["ultimate_winner"]
data_splits = results["data_splits"]

# Display Summary DataFrame
print("\n📊 MODEL PERFORMANCE SUMMARY:")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)

# Cross-validation scores details
print("\n📈 CROSS-VALIDATION SCORES (5-fold):")
print("="*70)
for model_name, cv_scores in raw_cv_scores.items():
    print(f"\n{model_name}:")
    print(f"  Scores: {[f'{score:.4f}' for score in cv_scores]}")
    print(f"  Mean:   {cv_scores.mean():.4f}")
    print(f"  Std:    {cv_scores.std():.4f}")

# Winner details
print("\n" + "="*70)
print("🏆 ULTIMATE WINNER:")
print("="*70)
if ultimate_winner is not None:
    print(f"✓ Best Model: {summary_df.iloc[0]['Model Name']}")
    print(f"✓ Test Accuracy: {summary_df.iloc[0]['Test Acc']}")
    print(f"✓ Test F1-Score: {summary_df.iloc[0]['Test F1']}")
    print(f"✓ Pipeline: {ultimate_winner}")
else:
    print("No winner determined.")

## Section 7️⃣: Export Best Model and Results

**Save everything you need for submission!**

In [ ]:
import joblib
from pathlib import Path

# Create output directory if it doesn't exist
Path(OUTPUT_PATH).mkdir(exist_ok=True)

# Step 1: Save the Best Model
print("💾 Exporting Results:")
print("="*70)

if ultimate_winner is not None:
    model_path = f"{OUTPUT_PATH}/best_model.pkl"
    joblib.dump(ultimate_winner, model_path)
    print(f"✓ Best model saved: {model_path}")

# Step 2: Export Summary Table
summary_path = f"{OUTPUT_PATH}/model_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"✓ Summary table saved: {summary_path}")

# Step 3: Save CV Scores
cv_summary = []
for model_name, cv_scores in raw_cv_scores.items():
    cv_summary.append({
        "Model": model_name,
        "CV_Mean": cv_scores.mean(),
        "CV_Std": cv_scores.std(),
        "CV_Min": cv_scores.min(),
        "CV_Max": cv_scores.max()
    })
cv_df = pd.DataFrame(cv_summary)
cv_path = f"{OUTPUT_PATH}/cv_scores.csv"
cv_df.to_csv(cv_path, index=False)
print(f"✓ CV scores saved: {cv_path}")

# Step 4: Create a comprehensive report
report = f"""
# ML TOOLKIT EVALUATION REPORT
Generated on Kaggle

## Dataset
- Shape: {X.shape[0]} samples × {X.shape[1]} features
- Target: {y.name}
- Class Distribution: {dict(y.value_counts())}

## Models Evaluated
{', '.join(models.keys())}

## Winner
Model: {summary_df.iloc[0]['Model Name'] if not summary_df.empty else 'N/A'}
Test Accuracy: {summary_df.iloc[0]['Test Acc'] if not summary_df.empty else 'N/A'}
Test F1-Score: {summary_df.iloc[0]['Test F1'] if not summary_df.empty else 'N/A'}

## Summary
{summary_df.to_string() if not summary_df.empty else 'No results'}

## CV Scores
{cv_df.to_string() if not cv_df.empty else 'No CV results'}
"""

report_path = f"{OUTPUT_PATH}/evaluation_report.txt"
with open(report_path, 'w') as f:
    f.write(report)
print(f"✓ Report saved: {report_path}")

print("\n✅ All results exported to /kaggle/output/")
print("\n📁 Files created:")
print("  • best_model.pkl - Your trained model")
print("  • model_summary.csv - Performance comparison table")
print("  • cv_scores.csv - Cross-validation details")
print("  • evaluation_report.txt - Full report")

## 🎓 Tips & Tricks for Using ML Toolkit on Kaggle

### 1. **Quick Tips**
- Always check your data! Missing values and outliers can hurt performance
- Use `test_size=0.3` for smaller datasets, `0.2` for larger ones
- Set `cv=5` for faster evaluation, `cv=10` for more robust estimates
- Use `verbose=True` during development, `verbose=False` for production

### 2. **Feature Engineering**
```python
# Before passing to evaluate_and_plot_models():
X['age_squared'] = X['age'] ** 2
X['log_fare'] = np.log1p(X['fare'])
# Then use this enhanced X
```

### 3. **Hyperparameter Tuning**
```python
# Expand param_grids for better tuning:
param_grids = {
    "Random Forest": {
        "model__n_estimators": [50, 100, 200],
        "model__max_depth": [5, 10, 15, None],
        "model__min_samples_split": [2, 5, 10]
    }
}
```

### 4. **Handling Class Imbalance**
```python
# For imbalanced data:
from imblearn.over_sampling import SMOTE
sampler = SMOTE(k_neighbors=5, random_state=42)
# Pass sampler=sampler to evaluate_and_plot_models()
```

### 5. **For Regression Tasks**
```python
# Change to regression:
results = evaluate_and_plot_models(
    ...,
    task_type="regression",  # ← Key change
    primary_metric="r2"      # or "mae", "rmse"
)
```

### 6. **Make Predictions on Test Data**
```python
# Load test data
test_df = pd.read_csv(f"{INPUT_PATH}/test.csv")
X_test_prepared = test_df[FEATURE_COLS]

# Use the best model to predict
best_model = results["ultimate_winner"]
predictions = best_model.predict(X_test_prepared)

# Save predictions
submission = pd.DataFrame({
    'id': test_df['id'],
    'prediction': predictions
})
submission.to_csv(f"{OUTPUT_PATH}/submission.csv", index=False)
```

### 7. **Common Issues & Solutions**

| Issue | Solution |
|-------|----------|
| `ModuleNotFoundError: my_ml_toolkit` | Restart kernel after installation |
| `ValueError: X and y must have same length` | Check X and y have same number of rows |
| `ConvergenceWarning` | Increase `max_iter` for model |
| `MemoryError` | Use `n_jobs=1` instead of `n_jobs=-1` |
| Slow evaluation | Reduce `cv` from 5 to 3, or use fewer models |

## 🎉 Summary: What You Just Learned!

You now have a complete end-to-end ML pipeline on Kaggle! 

### ✅ What the ML Toolkit Does:
1. 📊 **Loads and validates** your data
2. 🔀 **Automatically splits** data into train/test sets
3. 🤖 **Trains multiple models** in parallel
4. 🔍 **Performs hyperparameter tuning** with grid search
5. 📈 **Evaluates with cross-validation** (k-fold)
6. 🎨 **Generates visualizations** automatically
7. 🏆 **Ranks and compares** all models
8. 📊 **Extracts feature importance** for each model

### 🚀 Quick Start Template
```python
# 1. Install
pip install git+https://github.com/LAFFI01/MY_ML_TOOLS.git

# 2. Load data
X, y = prepare_data()  # Your data prep

# 3. Configure models
models = {"Model1": ..., "Model2": ...}

# 4. Evaluate
results = evaluate_and_plot_models(
    models=models,
    preprocess_pipeline=StandardScaler(),
    X=X, y=y,
    task_type="classification",
    verbose=True
)

# 5. Get results
best_model = results["ultimate_winner"]
summary = results["summary_df"]
```

### 📚 Need More Help?
- GitHub: https://github.com/LAFFI01/MY_ML_TOOLS
- Guide: https://github.com/LAFFI01/MY_ML_TOOLS/blob/main/FRIEND_GUIDE.md
- API Reference: Check the repository documentation

---

**Happy Machine Learning on Kaggle! 🚀**

*Created with ❤️ by LAFFI01*